In [1]:
from pathlib import Path

import paramiko
import pandas as pd
from tqdm.auto import tqdm

from utils import enrich_with_event_counts, BLIND_RECO_LOCATION


## Configs
#BLIND_RECO_LOCATION = "/mnt/cephfs/reco-blind/2020/data/recMay23/v5"

with open("./kharuk.txt") as f:
    PASSW = f.read()
JUMP_HOST = "localhost" # Forward to "lxui.jinr.ru"
JUMP_PORT = 2222
JUMP_USER = "kharuk"
JUMP_PASS = PASSW

TARGET_HOST = "10.220.31.220"
TARGET_USER = "kharuk"
TARGET_PASS = PASSW

def connect_to_target():
    # 1. Connect to jump host
    jump = paramiko.SSHClient()
    jump.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    jump.connect(JUMP_HOST, port=JUMP_PORT, username=JUMP_USER, password=JUMP_PASS)

    # 2. Open a tunnel channel through jump host to target
    jump_transport = jump.get_transport()
    dest_addr = (TARGET_HOST, 22)
    local_addr = ("127.0.0.1", 0)
    channel = jump_transport.open_channel("direct-tcpip", dest_addr, local_addr)

    # 3. Connect to target through the tunnel
    target = paramiko.SSHClient()
    target.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    target.connect(TARGET_HOST, username=TARGET_USER, password=TARGET_PASS, sock=channel)
    return jump, target

jump, target = None, None

In [2]:
# First observations

files_exp_reco_prelcuts = []
files_exp_reco_nocuts = []

jump, target = connect_to_target()
try:
    with target.open_sftp() as sftp:
        # 4. Use normally
        sftp = target.open_sftp()

        ## NUATM
        print("Head of BLIND_RECO data.")
        clusters = sorted([c for c in sftp.listdir(f"{BLIND_RECO_LOCATION}") if c.startswith('cluster')])
        print("Clusters:", clusters)

        for c in clusters:
            files_prelCuts = sorted([f"{BLIND_RECO_LOCATION}/{c}/{f}" for f in sftp.listdir(f"{BLIND_RECO_LOCATION}/{c}") if 'prelCuts' in f])
            files_all = sorted([f"{BLIND_RECO_LOCATION}/{c}/{f}" for f in sftp.listdir(f"{BLIND_RECO_LOCATION}/{c}") if 'prelCuts' not in f])
            files_exp_reco_prelcuts.append({
                'cluster': c,
                'remote_path': files_prelCuts,
                })
            files_exp_reco_nocuts.append({
                'cluster': c,
                'remote_path': files_all,
                })
finally:
    target.close()
    jump.close()

Exception (client): Error reading SSH protocol banner
Traceback (most recent call last):
  File "/home/albert/miniconda3/envs/baikal25/lib/python3.10/site-packages/paramiko/transport.py", line 2363, in _check_banner
    buf = self.packetizer.readline(timeout)
  File "/home/albert/miniconda3/envs/baikal25/lib/python3.10/site-packages/paramiko/packet.py", line 395, in readline
    buf += self._read_timeout(timeout)
  File "/home/albert/miniconda3/envs/baikal25/lib/python3.10/site-packages/paramiko/packet.py", line 673, in _read_timeout
    raise socket.timeout()
TimeoutError

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/albert/miniconda3/envs/baikal25/lib/python3.10/site-packages/paramiko/transport.py", line 2179, in run
    self._check_banner()
  File "/home/albert/miniconda3/envs/baikal25/lib/python3.10/site-packages/paramiko/transport.py", line 2367, in _check_banner
    raise SSHException(
paramiko.ssh_exception

SSHException: Error reading SSH protocol banner

In [3]:
import pandas as pd

df_prelcuts = pd.DataFrame(files_exp_reco_prelcuts)
df_prelcuts['particle_type'] = 'exo_reco_prelcuts'
df_prelcuts['part'] = 0

df_nocuts = pd.DataFrame(files_exp_reco_nocuts)
df_nocuts['particle_type'] = 'exo_reco_nocuts'
df_nocuts['part'] = 0

df = pd.concat([df_prelcuts, df_nocuts])
df_catalog = df.explode('remote_path')

In [4]:
df_catalog

,cluster,remote_path,particle_type,part
0,cluster1,/mnt/cephfs/reco-blind/2020/data/recMay23/v5//...,exo_reco_prelcuts,0
0,cluster1,/mnt/cephfs/reco-blind/2020/data/recMay23/v5//...,exo_reco_prelcuts,0
0,cluster1,/mnt/cephfs/reco-blind/2020/data/recMay23/v5//...,exo_reco_prelcuts,0
0,cluster1,/mnt/cephfs/reco-blind/2020/data/recMay23/v5//...,exo_reco_prelcuts,0
0,cluster1,/mnt/cephfs/reco-blind/2020/data/recMay23/v5//...,exo_reco_prelcuts,0
...,...,...,...,...
6,cluster7,/mnt/cephfs/reco-blind/2020/data/recMay23/v5//...,exo_reco_nocuts,0
6,cluster7,/mnt/cephfs/reco-blind/2020/data/recMay23/v5//...,exo_reco_nocuts,0
6,cluster7,/mnt/cephfs/reco-blind/2020/data/recMay23/v5//...,exo_reco_nocuts,0
6,cluster7,/mnt/cephfs/reco-blind/2020/data/recMay23/v5//...,exo_reco_nocuts,0


In [5]:
def download_one(target, remote_path: str, local_path: Path) -> tuple[str, bool, str]:
    """Open a dedicated SFTP channel, download one file, close the channel."""
    local_path.parent.mkdir(parents=True, exist_ok=True)
    if local_path.exists():
        return remote_path, True, ""
    try:
        sftp = target.open_sftp()
        try:
            sftp.get(remote_path, str(local_path))
        except Exception as e:
            raise e
        finally:
            sftp.close()
        return remote_path, True, ""
    except Exception as e:
        raise e
try:  
    jump, target = connect_to_target()
    for i, row in tqdm(df_catalog.iterrows(), total=len(df_catalog)):
        name = Path(row['remote_path']).name
        download_one(target, row['remote_path'], local_path=Path(f"./reco_blind_files/{name}"))
except Exception as e:
    raise e
finally:
    target.close()
    jump.close()

  0%|          | 0/3548 [00:00<?, ?it/s]

In [ ]:
target.close()
jump.close()

In [5]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def download_one(target, remote_path: str, local_path: Path) -> tuple[str, bool, str]:
    """Open a dedicated SFTP channel, download one file, close the channel."""
    local_path.parent.mkdir(parents=True, exist_ok=True)
    if local_path.exists():
        return remote_path, True, ""
    sftp = target.open_sftp()
    try:
        sftp.get(remote_path, str(local_path))
    except Exception as e:
        raise e
    finally:
        sftp.close()
    return remote_path, True, ""


N_WORKERS = 3  # keep below server's MaxSessions (usually 10)

jump, target = connect_to_target()
try:
    rows = [(Path(row['remote_path']).name, row['remote_path'])
            for _, row in df_catalog.iterrows()]

    with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
        futures = {
            pool.submit(download_one, target, rpath, Path(f"./reco_blind_files/{name}")): rpath
            for name, rpath in rows
        }
        for fut in tqdm(as_completed(futures), total=len(futures)):
            rpath, ok, err = fut.result()  # raises on error
finally:
    target.close()
    jump.close()

  0%|          | 0/3548 [00:00<?, ?it/s]

Secsh channel 15 open FAILED: open failed: Connect failed
Secsh channel 30 open FAILED: open failed: Connect failed
Secsh channel 41 open FAILED: open failed: Connect failed
Secsh channel 74 open FAILED: open failed: Connect failed
Secsh channel 139 open FAILED: open failed: Connect failed
Secsh channel 201 open FAILED: open failed: Connect failed
Secsh channel 227 open FAILED: open failed: Connect failed


KeyboardInterrupt: 